# Experiment 28: Leakage-Safe Ensemble Blend

This experiment focuses on getting stronger validation performance without the runtime of Experiment 27.

The expensive target/frequency encoding work is computed once per outer fold and reused across every model. Multiple diverse XGBoost configurations are then trained on the same leakage-safe features.

OOF predictions are saved and used for a large, cheap blend search.

The goal is to improve ROC-AUC while keeping runtime practical.


In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

RANDOM_STATE = 42
N_SPLITS = 3
SMOOTHING = 20.0

DATA_DIR = Path("../data")
SUBMISSION_DIR = Path("../submissions")
RESULTS_DIR = Path("../results")

SUBMISSION_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")

TARGET = "Will_Buy_EV"

y = train[TARGET].map({"No": 0, "Yes": 1}).astype(np.int8)

X = train.drop(columns=[TARGET]).copy()
X_test = test.copy()

print("Train:", X.shape)
print("Test:", X_test.shape)
print("Positive rate:", round(y.mean(), 6))


Train: (668665, 14)
Test: (286571, 14)
Positive rate: 0.174645


In [2]:
# Keep the feature recipe aligned with the strongest previous approach.

numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()

combined = pd.concat([X, X_test], axis=0, ignore_index=True)
combined_encoded = pd.get_dummies(
    combined,
    columns=categorical_cols,
    dummy_na=True
)

X_base = combined_encoded.iloc[:len(X)].copy()
X_test_base = combined_encoded.iloc[len(X):].copy()

X_test_base = X_test_base.reindex(columns=X_base.columns, fill_value=0)

IDENTITY_COLS = [
    "Age",
    "Annual_Income_USD",
    "Daily_Commute_km",
    "Number_of_Cars_Owned",
    "Charging_Stations_Near_Home",
    "Charging_Stations_Near_Work",
    "Environmental_Concern_Level",
]

PAIR_COLS = [
    ("Age", "Annual_Income_USD"),
    ("Age", "Daily_Commute_km"),
    ("Age", "Current_Car_Type"),
    ("Annual_Income_USD", "Current_Car_Type"),
    ("Annual_Income_USD", "City_Type"),
    ("Daily_Commute_km", "Current_Car_Type"),
    ("Charging_Stations_Near_Home", "Charging_Stations_Near_Work"),
    ("Environmental_Concern_Level", "Range_Anxiety_Level"),
]

IDENTITY_COLS = [c for c in IDENTITY_COLS if c in X.columns]
PAIR_COLS = [p for p in PAIR_COLS if all(c in X.columns for c in p)]

print("Base encoded features:", X_base.shape[1])
print("Identity features:", len(IDENTITY_COLS))
print("Pair features:", len(PAIR_COLS))


Base encoded features: 31
Identity features: 7
Pair features: 8


In [3]:
def add_group_features(
    train_raw,
    target_values,
    apply_raw,
    columns,
    smoothing=20.0,
):
    global_mean = float(target_values.mean())
    result = pd.DataFrame(index=apply_raw.index)

    for cols in columns:
        if isinstance(cols, str):
            cols_tuple = (cols,)
            name = cols
        else:
            cols_tuple = tuple(cols)
            name = "__".join(cols_tuple)

        stats = train_raw[list(cols_tuple)].copy()
        stats["_target_"] = np.asarray(target_values)

        grouped = (
            stats.groupby(list(cols_tuple), dropna=False)["_target_"]
            .agg(["mean", "count"])
            .reset_index()
        )

        grouped["encoded"] = (
            grouped["mean"] * grouped["count"]
            + global_mean * smoothing
        ) / (grouped["count"] + smoothing)

        encoded_map = grouped[
            list(cols_tuple) + ["encoded"]
        ]

        merged = apply_raw[list(cols_tuple)].merge(
            encoded_map,
            on=list(cols_tuple),
            how="left",
            sort=False,
        )

        result[f"TE__{name}"] = merged["encoded"].fillna(global_mean).to_numpy()

        freq = (
            stats.groupby(list(cols_tuple), dropna=False)
            .size()
            .rename("frequency")
            .reset_index()
        )

        freq_map = apply_raw[list(cols_tuple)].merge(
            freq,
            on=list(cols_tuple),
            how="left",
            sort=False,
        )

        result[f"FREQ__{name}"] = (
            freq_map["frequency"].fillna(0).to_numpy()
        )

    return result.reset_index(drop=True)


def build_outer_fold_features(
    outer_train_raw,
    outer_train_y,
    outer_valid_raw,
    inner_splits=3,
    smoothing=20.0,
):
    inner_cv = StratifiedKFold(
        n_splits=inner_splits,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    train_extra_parts = []

    first_columns = None

    for inner_train_idx, inner_valid_idx in inner_cv.split(
        outer_train_raw,
        outer_train_y
    ):
        inner_train_raw = outer_train_raw.iloc[inner_train_idx].reset_index(drop=True)
        inner_valid_raw = outer_train_raw.iloc[inner_valid_idx].reset_index(drop=True)
        inner_y = outer_train_y.iloc[inner_train_idx].reset_index(drop=True)

        inner_features = add_group_features(
            inner_train_raw,
            inner_y,
            inner_valid_raw,
            IDENTITY_COLS + PAIR_COLS,
            smoothing=smoothing,
        )

        if first_columns is None:
            first_columns = inner_features.columns.tolist()
            train_extra = pd.DataFrame(
                np.nan,
                index=np.arange(len(outer_train_raw)),
                columns=first_columns,
            )

        train_extra.iloc[inner_valid_idx] = inner_features.to_numpy()

    full_train_extra = add_group_features(
        outer_train_raw.reset_index(drop=True),
        outer_train_y.reset_index(drop=True),
        outer_train_raw.reset_index(drop=True),
        IDENTITY_COLS + PAIR_COLS,
        smoothing=smoothing,
    )

    valid_extra = add_group_features(
        outer_train_raw.reset_index(drop=True),
        outer_train_y.reset_index(drop=True),
        outer_valid_raw.reset_index(drop=True),
        IDENTITY_COLS + PAIR_COLS,
        smoothing=smoothing,
    )

    return (
        train_extra.reset_index(drop=True),
        valid_extra.reset_index(drop=True),
    )


In [4]:
# Precompute leakage-safe outer-fold matrices ONCE.
# Every model below reuses these matrices.

outer_cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

fold_data = []

print("Building leakage-safe fold features...")

for fold, (train_idx, valid_idx) in enumerate(
    outer_cv.split(X, y),
    start=1,
):
    print(f"Preparing fold {fold}/{N_SPLITS}...")

    X_train_raw = X.iloc[train_idx].reset_index(drop=True)
    X_valid_raw = X.iloc[valid_idx].reset_index(drop=True)

    y_train = y.iloc[train_idx].reset_index(drop=True)
    y_valid = y.iloc[valid_idx].reset_index(drop=True)

    train_extra, valid_extra = build_outer_fold_features(
        X_train_raw,
        y_train,
        X_valid_raw,
        inner_splits=3,
        smoothing=SMOOTHING,
    )

    X_train_fold = pd.concat(
        [
            X_base.iloc[train_idx].reset_index(drop=True),
            train_extra,
        ],
        axis=1,
    )

    X_valid_fold = pd.concat(
        [
            X_base.iloc[valid_idx].reset_index(drop=True),
            valid_extra,
        ],
        axis=1,
    )

    fold_data.append(
        {
            "train_idx": train_idx,
            "valid_idx": valid_idx,
            "X_train": X_train_fold.astype(np.float32),
            "X_valid": X_valid_fold.astype(np.float32),
            "y_train": y_train.to_numpy(),
            "y_valid": y_valid.to_numpy(),
        }
    )

print("Fold feature matrices ready.")
print("Features per fold:", fold_data[0]["X_train"].shape[1])


Building leakage-safe fold features...
Preparing fold 1/3...
Preparing fold 2/3...
Preparing fold 3/3...
Fold feature matrices ready.
Features per fold: 61


In [5]:
# Diverse but intentionally compact model pool.
# The objective is diversity, not dozens of nearly identical models.

MODEL_CONFIGS = {
    "XGB_A": dict(
        n_estimators=700,
        max_depth=4,
        learning_rate=0.035,
        min_child_weight=2,
        subsample=0.90,
        colsample_bytree=0.90,
        reg_alpha=0.0,
        reg_lambda=1.0,
    ),
    "XGB_B": dict(
        n_estimators=900,
        max_depth=5,
        learning_rate=0.030,
        min_child_weight=2,
        subsample=0.90,
        colsample_bytree=0.85,
        reg_alpha=0.0,
        reg_lambda=1.0,
    ),
    "XGB_C": dict(
        n_estimators=800,
        max_depth=6,
        learning_rate=0.030,
        min_child_weight=3,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.0,
        reg_lambda=1.5,
    ),
    "XGB_D": dict(
        n_estimators=650,
        max_depth=4,
        learning_rate=0.045,
        min_child_weight=4,
        subsample=0.95,
        colsample_bytree=0.80,
        reg_alpha=0.05,
        reg_lambda=1.5,
    ),
    "XGB_E": dict(
        n_estimators=1000,
        max_depth=5,
        learning_rate=0.025,
        min_child_weight=4,
        subsample=0.85,
        colsample_bytree=0.90,
        reg_alpha=0.0,
        reg_lambda=2.0,
    ),
    "XGB_F": dict(
        n_estimators=750,
        max_depth=7,
        learning_rate=0.025,
        min_child_weight=5,
        subsample=0.85,
        colsample_bytree=0.80,
        reg_alpha=0.10,
        reg_lambda=2.0,
    ),
    "XGB_G": dict(
        n_estimators=600,
        max_depth=3,
        learning_rate=0.050,
        min_child_weight=2,
        subsample=0.95,
        colsample_bytree=0.95,
        reg_alpha=0.0,
        reg_lambda=1.0,
    ),
    "XGB_H": dict(
        n_estimators=850,
        max_depth=6,
        learning_rate=0.025,
        min_child_weight=6,
        subsample=0.90,
        colsample_bytree=0.75,
        reg_alpha=0.20,
        reg_lambda=2.5,
    ),
}

print("Models:", len(MODEL_CONFIGS))


Models: 8


In [6]:
oof_predictions = pd.DataFrame(
    index=np.arange(len(X)),
    columns=list(MODEL_CONFIGS),
    dtype=np.float64,
)

fold_scores = []

for model_name, params in MODEL_CONFIGS.items():
    print(f"\n{'=' * 60}")
    print(f"Running {model_name}")
    print(f"{'=' * 60}")

    model_oof = np.zeros(len(X), dtype=np.float64)

    for fold_number, fold in enumerate(fold_data, start=1):
        model = XGBClassifier(
            **params,
            objective="binary:logistic",
            eval_metric="auc",
            tree_method="hist",
            random_state=RANDOM_STATE + fold_number,
            n_jobs=-1,
        )

        model.fit(
            fold["X_train"],
            fold["y_train"],
            eval_set=[(fold["X_valid"], fold["y_valid"])],
            verbose=False,
        )

        predictions = model.predict_proba(
            fold["X_valid"]
        )[:, 1]

        model_oof[fold["valid_idx"]] = predictions

        score = roc_auc_score(
            fold["y_valid"],
            predictions,
        )

        print(
            f"{model_name} | Fold {fold_number} | "
            f"ROC-AUC: {score:.6f}"
        )

    overall_score = roc_auc_score(y, model_oof)
    oof_predictions[model_name] = model_oof

    fold_scores.append(
        {
            "model": model_name,
            "oof_roc_auc": overall_score,
        }
    )

    print(
        f"{model_name} | OOF ROC-AUC: {overall_score:.6f}"
    )

model_results = (
    pd.DataFrame(fold_scores)
    .sort_values("oof_roc_auc", ascending=False)
    .reset_index(drop=True)
)

display(model_results)



Running XGB_A
XGB_A | Fold 1 | ROC-AUC: 0.944096
XGB_A | Fold 2 | ROC-AUC: 0.945296
XGB_A | Fold 3 | ROC-AUC: 0.945055
XGB_A | OOF ROC-AUC: 0.944758

Running XGB_B
XGB_B | Fold 1 | ROC-AUC: 0.943905
XGB_B | Fold 2 | ROC-AUC: 0.944767
XGB_B | Fold 3 | ROC-AUC: 0.944966
XGB_B | OOF ROC-AUC: 0.944401

Running XGB_C
XGB_C | Fold 1 | ROC-AUC: 0.943777
XGB_C | Fold 2 | ROC-AUC: 0.944605
XGB_C | Fold 3 | ROC-AUC: 0.944785
XGB_C | OOF ROC-AUC: 0.944186

Running XGB_D
XGB_D | Fold 1 | ROC-AUC: 0.944052
XGB_D | Fold 2 | ROC-AUC: 0.944928
XGB_D | Fold 3 | ROC-AUC: 0.945032
XGB_D | OOF ROC-AUC: 0.944570

Running XGB_E
XGB_E | Fold 1 | ROC-AUC: 0.943946
XGB_E | Fold 2 | ROC-AUC: 0.945059
XGB_E | Fold 3 | ROC-AUC: 0.945085
XGB_E | OOF ROC-AUC: 0.944581

Running XGB_F
XGB_F | Fold 1 | ROC-AUC: 0.943808
XGB_F | Fold 2 | ROC-AUC: 0.944559
XGB_F | Fold 3 | ROC-AUC: 0.944863
XGB_F | OOF ROC-AUC: 0.944280

Running XGB_G
XGB_G | Fold 1 | ROC-AUC: 0.944111
XGB_G | Fold 2 | ROC-AUC: 0.945328
XGB_G | Fold 3 

,model,oof_roc_auc
0,XGB_G,0.944815
1,XGB_A,0.944758
2,XGB_E,0.944581
3,XGB_D,0.944570
4,XGB_B,0.944401
5,XGB_H,0.944352
6,XGB_F,0.944280
7,XGB_C,0.944186


In [7]:
# Cheap blend search.
#
# Once OOF predictions exist, testing thousands of blends is extremely fast.
# We search:
#   - every pair
#   - every triple
#   - every 4-model combination
#   - multiple weight patterns
#
# No model retraining happens here.

blend_results = []

model_names = model_results["model"].tolist()

weight_patterns = [
    [0.50, 0.50],
    [0.60, 0.40],
    [0.70, 0.30],
    [0.75, 0.25],
    [0.80, 0.20],
]

for a, b in combinations(model_names, 2):
    for weights in weight_patterns:
        pred = (
            weights[0] * oof_predictions[a].to_numpy()
            + weights[1] * oof_predictions[b].to_numpy()
        )

        blend_results.append(
            {
                "type": "pair",
                "models": f"{a}+{b}",
                "weights": str(weights),
                "roc_auc": roc_auc_score(y, pred),
            }
        )

triple_weights = [
    [1/3, 1/3, 1/3],
    [0.50, 0.30, 0.20],
    [0.50, 0.25, 0.25],
    [0.60, 0.20, 0.20],
    [0.70, 0.15, 0.15],
]

for combo in combinations(model_names[:8], 3):
    for weights in triple_weights:
        pred = sum(
            w * oof_predictions[m].to_numpy()
            for w, m in zip(weights, combo)
        )

        blend_results.append(
            {
                "type": "triple",
                "models": "+".join(combo),
                "weights": str(weights),
                "roc_auc": roc_auc_score(y, pred),
            }
        )

quad_weights = [
    [0.40, 0.30, 0.20, 0.10],
    [0.35, 0.30, 0.20, 0.15],
    [0.25, 0.25, 0.25, 0.25],
    [0.50, 0.20, 0.15, 0.15],
]

for combo in combinations(model_names[:6], 4):
    for weights in quad_weights:
        pred = sum(
            w * oof_predictions[m].to_numpy()
            for w, m in zip(weights, combo)
        )

        blend_results.append(
            {
                "type": "quad",
                "models": "+".join(combo),
                "weights": str(weights),
                "roc_auc": roc_auc_score(y, pred),
            }
        )

blend_results = (
    pd.DataFrame(blend_results)
    .sort_values("roc_auc", ascending=False)
    .reset_index(drop=True)
)

print("Top blends:")
display(blend_results.head(20))


Top blends:


,type,models,weights,roc_auc
0,triple,XGB_G+XGB_A+XGB_F,"[0.6, 0.2, 0.2]",0.944872
1,triple,XGB_G+XGB_A+XGB_F,"[0.7, 0.15, 0.15]",0.944871
2,triple,XGB_G+XGB_A+XGB_F,"[0.5, 0.3, 0.2]",0.944870
3,triple,XGB_G+XGB_A+XGB_E,"[0.6, 0.2, 0.2]",0.944866
4,triple,XGB_G+XGB_A+XGB_E,"[0.5, 0.3, 0.2]",0.944865
5,triple,XGB_G+XGB_A+XGB_F,"[0.5, 0.25, 0.25]",0.944864
6,triple,XGB_G+XGB_A+XGB_E,"[0.5, 0.25, 0.25]",0.944863
7,triple,XGB_G+XGB_A+XGB_E,"[0.7, 0.15, 0.15]",0.944863
8,pair,XGB_G+XGB_A,"[0.6, 0.4]",0.944862
9,triple,XGB_G+XGB_E+XGB_F,"[0.7, 0.15, 0.15]",0.944862


In [8]:
# Also test rank averaging.
# This is often useful when individual models have different probability calibration.

rank_predictions = pd.DataFrame(index=oof_predictions.index)

for column in oof_predictions.columns:
    rank_predictions[column] = (
        oof_predictions[column]
        .rank(method="average", pct=True)
    )

rank_results = []

for r in range(2, min(7, len(model_names) + 1)):
    for combo in combinations(model_names[:8], r):
        pred = rank_predictions[list(combo)].mean(axis=1).to_numpy()

        rank_results.append(
            {
                "type": "rank_average",
                "models": "+".join(combo),
                "weights": "equal",
                "roc_auc": roc_auc_score(y, pred),
            }
        )

rank_results = (
    pd.DataFrame(rank_results)
    .sort_values("roc_auc", ascending=False)
    .reset_index(drop=True)
)

display(rank_results.head(20))


,type,models,weights,roc_auc
0,rank_average,XGB_G+XGB_A,equal,0.944860
1,rank_average,XGB_G+XGB_A+XGB_E,equal,0.944845
2,rank_average,XGB_G+XGB_A+XGB_F,equal,0.944835
3,rank_average,XGB_G+XGB_E,equal,0.944826
4,rank_average,XGB_G+XGB_A+XGB_D,equal,0.944821
5,rank_average,XGB_G+XGB_A+XGB_E+XGB_D,equal,0.944816
6,rank_average,XGB_G+XGB_A+XGB_E+XGB_F,equal,0.944814
7,rank_average,XGB_G+XGB_A+XGB_H,equal,0.944813
8,rank_average,XGB_G+XGB_A+XGB_D+XGB_F,equal,0.944812
9,rank_average,XGB_G+XGB_A+XGB_B,equal,0.944806


In [9]:
# Combine ordinary and rank blend candidates.

all_blends = pd.concat(
    [blend_results, rank_results],
    ignore_index=True,
)

all_blends = (
    all_blends
    .sort_values("roc_auc", ascending=False)
    .reset_index(drop=True)
)

all_blends.to_csv(
    RESULTS_DIR / "experiment_28_blend_results.csv",
    index=False,
)

model_results.to_csv(
    RESULTS_DIR / "experiment_28_model_results.csv",
    index=False,
)

oof_predictions.to_csv(
    RESULTS_DIR / "experiment_28_oof_predictions.csv",
    index=False,
)

print("Best Experiment 28 candidate:")
display(all_blends.head(10))


Best Experiment 28 candidate:


,type,models,weights,roc_auc
0,triple,XGB_G+XGB_A+XGB_F,"[0.6, 0.2, 0.2]",0.944872
1,triple,XGB_G+XGB_A+XGB_F,"[0.7, 0.15, 0.15]",0.944871
2,triple,XGB_G+XGB_A+XGB_F,"[0.5, 0.3, 0.2]",0.944870
3,triple,XGB_G+XGB_A+XGB_E,"[0.6, 0.2, 0.2]",0.944866
4,triple,XGB_G+XGB_A+XGB_E,"[0.5, 0.3, 0.2]",0.944865
5,triple,XGB_G+XGB_A+XGB_F,"[0.5, 0.25, 0.25]",0.944864
6,triple,XGB_G+XGB_A+XGB_E,"[0.5, 0.25, 0.25]",0.944863
7,triple,XGB_G+XGB_A+XGB_E,"[0.7, 0.15, 0.15]",0.944863
8,pair,XGB_G+XGB_A,"[0.6, 0.4]",0.944862
9,triple,XGB_G+XGB_E+XGB_F,"[0.7, 0.15, 0.15]",0.944862


In [ ]:
# Final training.
#
# Select the best ordinary probability blend unless rank averaging wins.
# The final submission is produced from models participating in the winning
# candidate.

best = all_blends.iloc[0]

print("Selected blend:")
print(best.to_dict())

selected_models = best["models"].split("+")
blend_type = best["type"]

if blend_type == "rank_average":
    selected_weights = [1.0 / len(selected_models)] * len(selected_models)
else:
    selected_weights = eval(best["weights"])

print("Models:", selected_models)
print("Weights:", selected_weights)

# Build leakage-safe full-training features.
full_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

train_extra_parts = []
feature_columns = None

for inner_train_idx, inner_valid_idx in full_cv.split(X, y):
    inner_train_raw = X.iloc[inner_train_idx].reset_index(drop=True)
    inner_valid_raw = X.iloc[inner_valid_idx].reset_index(drop=True)
    inner_y = y.iloc[inner_train_idx].reset_index(drop=True)

    encoded_valid = add_group_features(
        inner_train_raw,
        inner_y,
        inner_valid_raw,
        IDENTITY_COLS + PAIR_COLS,
        smoothing=SMOOTHING,
    )

    if feature_columns is None:
        feature_columns = encoded_valid.columns.tolist()

    encoded_valid = encoded_valid.reindex(columns=feature_columns)

    fold_frame = pd.DataFrame(
        {
            "row_idx": inner_valid_idx,
        }
    )

    for column in feature_columns:
        fold_frame[column] = encoded_valid[column].to_numpy()

    train_extra_parts.append(fold_frame)

# Reassemble the encoded features in the original training-row order.
full_train_extra = (
    pd.concat(train_extra_parts, axis=0)
    .sort_values("row_idx")
    .drop(columns=["row_idx"])
    .reset_index(drop=True)
)

full_train_extra = full_train_extra.reindex(columns=feature_columns)

# Safety check: every training row must have an encoded value.
if len(full_train_extra) != len(X):
    raise RuntimeError(
        f"Full-training feature row count mismatch: "
        f"{len(full_train_extra)} != {len(X)}"
    )

# Any unexpected missing values are filled with the global positive rate.
full_train_extra = full_train_extra.fillna(float(y.mean()))

# Encode the test set using the complete training data.
full_test_extra = add_group_features(
    X.reset_index(drop=True),
    y.reset_index(drop=True),
    X_test.reset_index(drop=True),
    IDENTITY_COLS + PAIR_COLS,
    smoothing=SMOOTHING,
)

full_test_extra = full_test_extra.reindex(columns=feature_columns)
full_test_extra = full_test_extra.fillna(float(y.mean()))

X_full = pd.concat(
    [
        X_base.reset_index(drop=True),
        full_train_extra.reset_index(drop=True),
    ],
    axis=1,
).astype(np.float32)

X_final_test = pd.concat(
    [
        X_test_base.reset_index(drop=True),
        full_test_extra.reset_index(drop=True),
    ],
    axis=1,
).astype(np.float32)

print("Final train shape:", X_full.shape)
print("Final test shape:", X_final_test.shape)
print("Final feature count:", X_full.shape[1])
print("Missing train values:", int(X_full.isna().sum().sum()))
print("Missing test values:", int(X_final_test.isna().sum().sum()))

Selected blend:
{'type': 'triple', 'models': 'XGB_G+XGB_A+XGB_F', 'weights': '[0.6, 0.2, 0.2]', 'roc_auc': 0.9448718803413296}
Models: ['XGB_G', 'XGB_A', 'XGB_F']
Weights: [0.6, 0.2, 0.2]


ValueError: DataFrame constructor not properly called!

In [ ]:
# Train only the models actually used by the winning blend.

test_predictions = {}

for model_name in selected_models:
    params = MODEL_CONFIGS[model_name]

    print(f"Training final {model_name}...")

    model = XGBClassifier(
        **params,
        objective="binary:logistic",
        eval_metric="auc",
        tree_method="hist",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

    model.fit(
        X_full,
        y,
        verbose=False,
    )

    test_predictions[model_name] = model.predict_proba(
        X_final_test
    )[:, 1]

if blend_type == "rank_average":
    ranked = []

    for model_name in selected_models:
        ranked.append(
            pd.Series(test_predictions[model_name])
            .rank(method="average", pct=True)
            .to_numpy()
        )

    final_prediction = np.mean(ranked, axis=0)

else:
    final_prediction = np.zeros(len(X_test))

    for model_name, weight in zip(
        selected_models,
        selected_weights,
    ):
        final_prediction += (
            weight * test_predictions[model_name]
        )

submission = pd.DataFrame({
    "id": test["id"],
    TARGET: final_prediction,
})

submission_path = SUBMISSION_DIR / "experiment_28.csv"
submission.to_csv(submission_path, index=False)

print(f"Saved: {submission_path}")
print(submission.head())
print("Prediction range:", float(final_prediction.min()), float(final_prediction.max()))


## Experiment 28 Summary

This experiment uses:

- leakage-safe nested OOF target encoding
- frequency encoding
- identity and pair features
- eight deliberately different XGBoost configurations
- OOF prediction storage
- pair, triple, and quadruple probability blending
- rank averaging
- final retraining only for models used by the winning blend

The expensive feature construction is performed once per fold rather than once per model.
